**Levene's Test**

Determine if variances across the groups are roughly equal

In [ ]:
!pip install scipy
!pip install scikit_posthocs

In [ ]:
from scipy.stats import levene, kruskal
from google.colab import drive
import csv
import os

drive.mount('/content/drive')

def get_scores(filename: str):
  output = []

  with open(filename, 'r') as file:
    reader = csv.reader(file)
    header = next(reader)
    try:
      column_index = header.index("aggregate_score")
    except ValueError:
      return -1

    for row in reader:
      aggregate_score = float(row[column_index])
      output.append(aggregate_score)

  return output


gpt4o_scores = get_scores("/content/drive/MyDrive/ResearchPapers/Statistical Analysis/scores/4o_judging_aggregated_scores.csv")
gemini_scores = get_scores("/content/drive/MyDrive/ResearchPapers/Statistical Analysis/scores/gemini_judging_aggregated_scores.csv")
claude_scores = get_scores("/content/drive/MyDrive/ResearchPapers/Statistical Analysis/scores/claude_judging_aggregated_scores.csv")
o3mini_scores = get_scores("/content/drive/MyDrive/ResearchPapers/Statistical Analysis/scores/o3mini_judging_aggregated_scores.csv")

# Run Levene's test (center='median' is more robust to non-normal data)
stat, p_value = levene(gpt4o_scores, gemini_scores, claude_scores, o3mini_scores, center='median')

# Print results
print(f"Levene's test statistic: {stat:.4f}, p-value: {p_value:.4f}")

# Interpretation
alpha = 0.05
if p_value > alpha:
    print("Fail to reject H₀: Variances are equal (homogeneity of variance holds)")
else:
    print("Reject H₀: Variances are not equal (consider Welch's ANOVA or Kruskal-Wallis)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Levene's test statistic: 1.6151, p-value: 0.1853
Fail to reject H₀: Variances are equal (homogeneity of variance holds)


**Kruskal-Wallis H-Test for Multiple Groups**

Determine if there is a statistically significant difference somewhere in the group of LLMs.

In [ ]:
# Perform Kruskal-Wallis test
statistic, p_value = kruskal(gpt4o_scores, gemini_scores, claude_scores, o3mini_scores)

# Print results
print(f"Kruskal-Wallis H-test statistic: {statistic:.4f}, p-value: {p_value:.4f}")

# Interpretation
alpha = 0.05
if p_value > alpha:
    print("Fail to reject H₀: No significant difference between groups")
else:
    print("Reject H₀: Significant difference between at least two groups")


Kruskal-Wallis H-test statistic: 21.0695, p-value: 0.0001
Reject H₀: Significant difference between at least two groups


In [ ]:
# prompt: now run a post-hoc Dunn’s test with Bonferroni correction to determine which pairs of models are significantly different

!pip install scikit_posthocs

import scikit_posthocs as sp
import pandas as pd
import numpy as np

# Combine the data into a single DataFrame
data = {
    'gpt4o': gpt4o_scores,
    'gemini': gemini_scores,
    'claude': claude_scores,
    'o3mini': o3mini_scores
}
df = pd.DataFrame(data)

# Melt the DataFrame to long format
df_melted = pd.melt(df, var_name='Model', value_name='Score')

# Perform Dunn's test with Bonferroni correction
dunn_result = sp.posthoc_dunn(df_melted, val_col='Score', group_col='Model', p_adjust='bonferroni')

# Print the results
dunn_result


,claude,gemini,gpt4o,o3mini
claude,1.000000,0.000072,0.030039,0.921461
gemini,0.000072,1.000000,0.698303,0.019083
gpt4o,0.030039,0.698303,1.000000,1.000000
o3mini,0.921461,0.019083,1.000000,1.000000


In [ ]:
import pandas as pd

print("Models with a statistically significant difference:")
alpha = 0.05
significant_pairs = []
for model1 in dunn_result.columns:
    for model2 in dunn_result.index:
        if model1 != model2 and (model2, model1) not in significant_pairs :
            p_value = dunn_result.loc[model2, model1]
            if p_value < alpha:
                print(f"{model1} vs {model2}: p-value = {p_value:.4f}")
                significant_pairs.append((model1, model2))

print("\n")

print("Models without a statistically significant difference:")
non_significant_pairs = []

for model1 in dunn_result.columns:
    for model2 in dunn_result.index:
      if model1 != model2 and (model2, model1) not in non_significant_pairs and (model1, model2) not in significant_pairs:
        p_value = dunn_result.loc[model2, model1]
        if p_value >= alpha:
          print(f"{model1} vs {model2}: p-value = {p_value:.4f}")
          non_significant_pairs.append((model1,model2))


Models with a statistically significant difference:
claude vs gemini: p-value = 0.0001
claude vs gpt4o: p-value = 0.0300
gemini vs o3mini: p-value = 0.0191


Models without a statistically significant difference:
claude vs o3mini: p-value = 0.9215
gemini vs gpt4o: p-value = 0.6983
gpt4o vs o3mini: p-value = 1.0000


Power Analysis

In [ ]:
H_stat = statistic  # kruskal-Wallis test statistic from previous test
k = 4  # number of LLMs
N = 101 * 4  # number of data points

# xompute epsilon squared (effect size)
epsilon_squared = (H_stat - k + 1) / (N - k)

print(f"Epsilon squared effect size: {epsilon_squared:.4f}")


Epsilon squared effect size: 0.0452


In [ ]:
from statsmodels.stats.power import FTestAnovaPower
import numpy as np

# convert epsilon squared to Cohen's f (approximation)
cohen_f = np.sqrt(epsilon_squared / (1 - epsilon_squared))

power_analysis = FTestAnovaPower()
alpha = 0.05
sample_size_per_group = 101
num_groups = 4
total_n = sample_size_per_group * num_groups

power = power_analysis.power(effect_size=cohen_f, nobs=total_n, alpha=alpha, k_groups=num_groups)

print(f"Achieved Power: {power:.4f}")


Achieved Power: 0.9674
